# TeCo: individual trials versus averages of up to 40

**CPU → Run all.** Use **Zyion** for both Drive prompts. No GPU needed.

- **TeCo task 1 only**, positive versus negative, the original lexical/contextual-word agreement labels, restored by event ID from the completed run’s checksum-verified true-label roster. No annotation CSV lookup or relabeling. Function words follow the existing selection policy. No new labels or EEG extraction.
- Reuse the completed `classification_similarity` feature caches. Input: **48 voltage features**, six scalp regions × eight 100-ms bins spanning **0–800 ms**. These are time-domain features, not frequency bands. This differs from the later ZuCo paper-style all-channel 224–304 ms setup. No paper regression cleaning is added.
- Fixed regularized **logistic regression**. Compare one trial per example against **up to 40 same-class trials**, within the same reader. Three separate held-out sentence, word-identity and reader evaluations. Only the named identity is excluded. Word splits purge overlapping training EEG windows.
- **Split first, average separately in training and testing.** No original event crosses the split. Smaller final groups are retained, and actual sizes are shown. Test labels form the averages, so the result describes class-grouped separation rather than single-word prediction.
- Controls: chance/dummy, pre-onset EEG and word/fixation properties. Scaling and imputation fit training averages only. No neutral downsampling or hyperparameter search. Scores are descriptive; no significance claim.

**Storage:** at most **10 MB total new Drive output**, under `Thesis/Results/word_sentiment_eeg/teco_average40/`. No new feature cache or EEG copy. Dependencies stay on temporary Colab disk. If a required cache is missing, stop rather than recomputing it. Completed cases are checked and reused. Do not change an active notebook's runtime to run this one.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


## 1. Connect and verify the saved inputs


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/teco_average40'
REPO=Path('/content/word-sentiment-eeg-teco-average40')
CODE_REVISION='c2b9fe78645fdef89e58c51d688ad875b473850e'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
# Temporary cloud-only scientific environment: no base-runtime upgrades/restart.
# BOOTSTRAP_START: also exercised verbatim in cloud CI.
# Use a file-rendering backend in subprocesses, not Colab's unavailable inline plugin.
os.environ['MPLBACKEND']='Agg'
ENV=Path('/content/word-eeg-paper-classification-python')
PYTHON=ENV/'bin/python'
def setup_command(command):
    print('SETUP:', ' '.join(map(str,command)), flush=True)
    result=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Scientific environment setup failed; the full diagnostic is printed above. No EEG processing started.')
# Colab may lack ensurepip. This also repairs a partially created environment.
setup_command([sys.executable,'-m','venv','--without-pip',str(ENV)])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'install',
               '--only-binary=:all:','-r',str(REPO/'requirements-paper-classification.txt')])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'check'])
setup_command([str(PYTHON),'-c',
    'import numpy,pandas,scipy,sklearn,matplotlib; '
    'print("Scientific environment ready:",numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,matplotlib.__version__)'])
# BOOTSTRAP_END
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,10_000_000-used):
    raise RuntimeError('Reserve at most 10 MB for results. This notebook never reads EEG signals.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 10 MB.')
print('New output folder:',OUT)


## 2. Run the comparisons
Reads the 12 existing TeCo feature caches. No EEG shard loading or feature preparation.


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[str(PYTHON),'-u',str(REPO/'scripts/teco_average40.py'),
         '--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='TeCo individual versus max40: saved features only')


## 3. Classification and actual averaging sizes
All predictions are out of fold. An average of several trials counts as one test example.


In [ ]:
scores=pd.read_csv(OUT/'scores.csv')
main=scores[scores.feature.eq('post')].copy()
for name in ['accuracy','balanced_accuracy','macro_f1','roc_auc']:
    main[name]=(main[name]*100).round(1)
display(main[['split','maximum_trials','model','balanced_accuracy','accuracy',
              'test_examples','actual_mean_trials','actual_min_trials','full40_test_examples']])
print('Primary comparison: logistic maximum_trials=40 versus logistic maximum_trials=1 and dummy.')
print('Chance BA: 50%. actual_mean_trials is the actual average size, not the requested maximum.')
display(pd.read_csv(OUT/'input_counts.csv'))
status=json.loads((OUT/'complete.json').read_text())
print('Run status:',status['status'],'| Completed cases:',status['cases'],'/18')


## 4. Controls and saved outputs


In [ ]:
controls=scores[scores.maximum_trials.eq(40)].copy()
controls['BA_percent']=(100*controls.balanced_accuracy).round(1)
display(controls[['split','feature','model','BA_percent','test_examples']])
print('post = post-onset EEG; pre = −200 to 0 ms EEG; nuisance = word/fixation properties only.')
print('If controls also separate classes, interpretation as sentiment-specific EEG needs caution.')
print('Saved predictions, averaging memberships, splits and per-reader checks:',OUT)
